# Raster: windows, mosaics, regridding, zone masks, atomic writes

A raster is `(array, BlockGrid)`. This notebook writes a synthetic block to a temporary folder
next to the notebook, reads it back in pieces and resamples it. See `docs/memory.md` for what
each operation allocates.

In [ ]:
import shutil
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from affine import Affine
from rasterio.windows import Window
from shapely.geometry import box

from idd_spatial_analysis.constants import WGS84, WORLD_CYLINDRICAL_EQUAL_AREA
from idd_spatial_analysis.frame import BlockGrid, grid_from_bounds
from idd_spatial_analysis.raster import (
    NearestIndexMap,
    mosaic,
    rasterize_zones,
    read_window,
    reproject_to_grid,
    resample_masked,
    write_geotiff_atomic,
)

WORK = Path.cwd() / "_raster_vignette_tmp"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

In [ ]:
# A 64 x 64 block of 100 m pixels with a NaN corner and a zero band, written atomically.
grid = BlockGrid(
    crs=WORLD_CYLINDRICAL_EQUAL_AREA,
    transform=Affine(100.0, 0, 0, 0, -100.0, 6400.0),
    height=64,
    width=64,
)
rng = np.random.default_rng(0)
population = rng.gamma(2.0, 50.0, size=grid.shape).astype("float32")
population[:, 20:24] = 0.0
population[:8, :8] = np.nan
path = write_geotiff_atomic(WORK / "population.tif", population, grid, nodata=np.nan)
print(path, sorted(p.name for p in WORK.iterdir()))

In [ ]:
# Read exactly the pixels covering some bounds; boundless reads pad the outside with NaN.
part, sub = read_window(path, (1000.0, 1000.0, 2000.0, 2000.0))
print(part.shape, sub.bounds)
edge, edge_grid = read_window(path, (-500.0, -500.0, 500.0, 500.0), boundless=True)
print(edge.shape, np.isnan(edge).sum(), "NaN pixels outside the raster")

In [ ]:
# Mosaics: two tiles back into one array over the requested bounds.
left_grid = grid.sub(Window(0, 0, 32, 64))
right_grid = grid.sub(Window(32, 0, 32, 64))
write_geotiff_atomic(WORK / "left.tif", population[:, :32], left_grid, nodata=np.nan)
write_geotiff_atomic(WORK / "right.tif", population[:, 32:], right_grid, nodata=np.nan)
merged, merged_grid = mosaic([WORK / "left.tif", WORK / "right.tif"], grid.bounds)
print(merged_grid.shape, np.array_equal(merged, population, equal_nan=True))

In [ ]:
# Pixel-centre nearest onto a coarser grid, built once and applied to a stack of slices.
coarse = BlockGrid(
    crs=grid.crs, transform=Affine(400.0, 0, 0, 0, -400.0, 6400.0), height=16, width=16
)
lookup = NearestIndexMap.build(grid, coarse)
stack = np.stack([population, 2 * population])
print(lookup.apply(stack).shape)

In [ ]:
# Compute only at the target pixels you care about (the populated ones, say).
wanted = rng.random(coarse.shape) < 0.25
sparse = resample_masked(population, grid, coarse, wanted)
print(wanted.sum(), "pixels computed;", np.isnan(sparse).sum(), "left as NaN")

In [ ]:
# Warps through rasterio for anything else: here an average onto a lon/lat grid.
lonlat = grid_from_bounds(grid.lonlat_bounds(), 0.002, WGS84)
averaged = reproject_to_grid(population, grid, lonlat, resampling="average")
print(lonlat.shape, np.isfinite(averaged).sum(), "finite pixels")

In [ ]:
# Zone polygons to an int32 mask (0 = no zone) with a window per zone.
zones = gpd.GeoDataFrame(
    {"zone": np.array([1, 2, 3, 4], dtype="int32")},
    geometry=[
        box(0, 3200, 3200, 6400),
        box(3200, 3200, 6400, 6400),
        box(0, 0, 3200, 3200),
        box(3200, 0, 6400, 3200),
    ],
    crs=grid.crs,
)
mask, windows = rasterize_zones(zones, "zone", grid)
print(np.unique(mask, return_counts=True))
print(windows[1])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(population, extent=(0, 6400, 0, 6400))
axes[0].set_title("population (100 m)")
axes[1].imshow(lookup.apply(population), extent=(0, 6400, 0, 6400))
axes[1].set_title("nearest, 400 m")
axes[2].imshow(mask, extent=(0, 6400, 0, 6400))
axes[2].set_title("zone mask")
plt.show()

In [ ]:
shutil.rmtree(WORK)